In [1]:
import os
import glob
import numpy as np
import math
import h5py
import joblib
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin

import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [2]:
class EEGVideoDataset(Dataset):
    def __init__(self, eeg_features, labels, video_features, video_ids):
        self.eeg_features = torch.tensor(eeg_features, dtype=torch.float32)
        self.video_features = torch.tensor(video_features, dtype=torch.float32)
        
        id_to_local_idx = {vid: idx for idx, vid in enumerate(video_ids)}
        local_labels = [id_to_local_idx[vid] for vid in labels]
        
        self.labels = torch.tensor(local_labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.eeg_features)
        
    def __getitem__(self, idx):
        eeg = self.eeg_features[idx]
        local_vid_idx = self.labels[idx]
        video = self.video_features[local_vid_idx]
        return eeg, video, local_vid_idx

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim)
        )
    def forward(self, x):
        return x + self.block(x)
    
class ClipModel(nn.Module):
    def __init__(self, eeg_dim: int, video_dim: int, projection_dim: int, 
                 dropout: float, temperature: float):
        super().__init__()
        self.eeg_dim = eeg_dim
        self.video_dim = video_dim
        hidden_dim = 256
        self.eeg_projection = nn.Sequential(
            nn.Linear(eeg_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            ResidualBlock(hidden_dim, dropout),
            nn.Linear(hidden_dim, projection_dim)
        )
        self.video_projection = nn.Sequential(
            nn.Linear(video_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            ResidualBlock(hidden_dim, dropout),
            nn.Linear(hidden_dim, projection_dim)
        )
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / temperature)))
    
    def forward(self, eeg_features, video_features):
        eeg_embeddings = F.normalize(self.eeg_projection(eeg_features), dim=-1)
        video_embeddings = F.normalize(self.video_projection(video_features), dim=-1)
        return eeg_embeddings, video_embeddings

In [4]:
def compute_multi_positive_contrastive_loss(eeg_emb, video_emb, vid_ids, logit_scale):
    logit_scale = logit_scale.exp()
    
    logits_eeg_to_video = logit_scale * torch.matmul(eeg_emb, video_emb.t())
    logits_video_to_eeg = logit_scale * torch.matmul(video_emb, eeg_emb.t())

    match_mask = (vid_ids.unsqueeze(1) == vid_ids.unsqueeze(0)).float()
    row_sum = match_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    match_mask_normalized = match_mask / row_sum
    
    log_probs_eeg = F.log_softmax(logits_eeg_to_video, dim=-1)
    log_probs_video = F.log_softmax(logits_video_to_eeg, dim=-1)
    
    loss_eeg = -(match_mask_normalized * log_probs_eeg).sum(dim=1).mean()
    loss_video = -(match_mask_normalized * log_probs_video).sum(dim=1).mean()
    
    return (loss_eeg + loss_video) / 2.0

In [5]:
class CLIP_Classifier:
    def __init__(self, projection_dim: int = 256, epochs: int = 15, batch_size: int = 32, 
                 lr: int = 1e-3, weight_decay: float = 1e-4, dropout: float = 0.1, 
                 temperature: float = 0.07, device: str = "cpu"):
        self.projection_dim = projection_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.dropout = dropout
        self.temperature = temperature
        self.device = device
        
        self.model = None
        self.eeg_scaler = StandardScaler()

    def fit(self, X_train, Z_train):
        # 1. Scale EEG and video features
        X_scaled = self.eeg_scaler.fit_transform(np.asarray(X_train, dtype=np.float32))
        Z_scaled = (np.asarray(Z_train, dtype=np.float32))

        # 2. Extract unique video features and map sample indices for multi-positive contrastive loss
        unique_videos, inverse_indices = np.unique(Z_scaled, axis=0, return_inverse=True)
        unique_video_ids = np.arange(len(unique_videos))
        labels = inverse_indices

        eeg_dim = X_scaled.shape[1]
        video_dim = unique_videos.shape[1]

        # 3. Setup dataset and dataloader
        dataset = EEGVideoDataset(X_scaled, labels, unique_videos, unique_video_ids)
        drop_last = len(dataset) >= self.batch_size
        train_loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True, drop_last=drop_last)

        # 4. Initialize PyTorch CLIP model and optimizer
        self.model = ClipModel(
            eeg_dim=eeg_dim,
            video_dim=video_dim,
            projection_dim=self.projection_dim,
            dropout=self.dropout,
            temperature=self.temperature
        ).to(self.device)

        optimizer = optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        # 5. Training Loop
        self.model.train()
        for epoch in range(self.epochs):
            for eeg, video, vid_ids in train_loader:
                eeg = eeg.to(self.device)
                video = video.to(self.device)
                vid_ids = vid_ids.to(self.device)

                optimizer.zero_grad()
                eeg_emb, video_emb = self.model(eeg, video)
                loss = compute_multi_positive_contrastive_loss(
                    eeg_emb, video_emb, vid_ids, self.model.logit_scale.clamp(max=100.0)
                )
                loss.backward()
                optimizer.step()

        return self

    def get_score_matrix(self, X_test, candidate_video_features):
        """Computes the similarity score matrix across test EEG samples and candidate videos."""
        self.model.eval()
        
        X_test_scaled = self.eeg_scaler.transform(np.asarray(X_test, dtype=np.float32))
        cand_scaled = (np.asarray(candidate_video_features, dtype=np.float32))

        eeg_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=self.device)
        cand_tensor = torch.tensor(cand_scaled, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            eeg_emb = F.normalize(self.model.eeg_projection(eeg_tensor), dim=-1)
            video_emb = F.normalize(self.model.video_projection(cand_tensor), dim=-1)
            logit_scale = self.model.logit_scale.exp().clamp(max=100.0)
            score_matrix = (eeg_emb @ video_emb.T * logit_scale).cpu().numpy()

        return score_matrix

In [6]:
BASE_DIR     = Path('../eeg-mind-reading-challenge')
STIM_DIR     = BASE_DIR / 'stimuli_train'  
FEAT_DIR     = BASE_DIR / 'features'       
TEMP_FEAT    = BASE_DIR / 'temp_features' 
TRAIN_DIR    = BASE_DIR / 'trn'
ARTIFACT_DIR = Path('../artifacts')
H5_DIR       = Path('../eeg_embeddings')  # TODO: Change this where pre-processed eeg signals are extracted to

os.makedirs(ARTIFACT_DIR, exist_ok=True)

SFREQ = 500
DEC = 5
N_FRAMES = 8

def features_for(ids, layer_path):
    feat_path = FEAT_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

vids = sorted(list(STIM_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_DIR}")
all_stim_ids = np.arange(1, len(vids) + 1)

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
sc = StandardScaler().fit(X_unique_alex)
X_trn_alex_z = sc.transform(X_unique_alex)
sc2 = StandardScaler().fit(X_unique_dino)
X_trn_dino_z = sc2.transform(X_unique_dino)
sc3 = StandardScaler().fit(X_unique_clip)
X_trn_clip_z = sc3.transform(X_unique_clip)

X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])

pca = PCA(n_components=0.95, svd_solver='full', random_state=42).fit(X_trn_ensemble)
X_unique_p = pca.transform(X_trn_ensemble)

sc4 = StandardScaler().fit(X_unique_p)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

joblib.dump({
    'sc': sc, 'pca': pca, 'sc2': sc2, 'sc3': sc3, 'sc4': sc4}, 
    ARTIFACT_DIR / 'clip_sub-global.joblib', compress=3)


Found 1000 total videos in ../eeg-mind-reading-challenge/stimuli_train
Loading all unique video features for global PCA...
Preprocessing unique video features...
Video feature map successfully created!


['../artifacts/clip_sub-global.joblib']

In [7]:
# --- 2. Loop Through Each Subject Individually ---
embedding_sources = ['schubki_cbramod','schubki_labram', 'schubki_luna', 'schubki_labram_special']
for sub in range(1, 7):  
    print(f"\n==============================")
    print(f"Training Models for Subject {sub:02d}")
    print(f"==============================")
    
    subject_models = {}
    score_matrices = []
    Y_train_ref = None

    candidate_matrix = np.array([video_feature_map[int(sid)] for sid in all_stim_ids])

    for source_name in embedding_sources:
        h5_path = H5_DIR / source_name / f'sub-{sub}.h5'
        if not os.path.exists(h5_path):
            print(f"-> Skipping source '{source_name}' for Subject {sub} (file not found at {h5_path})")
            continue
            
        print(f"-> Training CLIP model on source: {source_name}")

        with h5py.File(h5_path, 'r') as f:
            if 'X' in f and 'Y' in f:
                X_train = np.array(f['X'], dtype=np.float32)
                Y_train = np.array(f['Y'], dtype=int)

        print(X_train.shape)

        Y_train_ref = Y_train
        Z_train = np.array([video_feature_map[int(vid)] for vid in Y_train])

        model = CLIP_Classifier(projection_dim=128, epochs=10, batch_size=32, lr=5e-4, weight_decay=1e-2, dropout=0.3)
        model.fit(X_train, Z_train)

        score_matrix = model.get_score_matrix(X_train, candidate_matrix)
        score_matrices.append(score_matrix)
        
        subject_models[source_name] = model
        
        order_idx = np.argsort(-score_matrix, axis=1)
        ranked = all_stim_ids[order_idx].astype(np.int32)
        top1 = (ranked[:, 0] == Y_train).mean()
        top5 = (ranked[:, :5] == Y_train[:, None]).any(axis=1).mean()
        print(f"   [{source_name}] Train -> Top-1: {top1:.4f} | Top-5: {top5:.4f}")

    if not score_matrices:
        print(f"Skipping Subject {sub:02d} (no valid embedding sources found)")
        continue

    # --- SCORE-LEVEL ENSEMBLING ---
    ensemble_score_matrix = np.mean(score_matrices, axis=0)
    ensemble_order_idx = np.argsort(-ensemble_score_matrix, axis=1)
    ensemble_ranked = all_stim_ids[ensemble_order_idx].astype(np.int32)
    
    ensemble_top1 = (ensemble_ranked[:, 0] == Y_train_ref).mean()
    ensemble_top5 = (ensemble_ranked[:, :5] == Y_train_ref[:, None]).any(axis=1).mean()
    print(f"\n🌟 Ensembled Subject {sub:02d} Train -> Top-1: {ensemble_top1:.4f} | Top-5: {ensemble_top5:.4f}")

    # Save all models for this subject as a bundle artifact
    joblib.dump({'models': subject_models, 'sources': list(subject_models.keys())}, 
                ARTIFACT_DIR / f'clip_ensemble_sub-{sub:02d}.joblib', compress=3)
    print(f"Saved ensembled CLIP models for Subject {sub:02d}!")


Training Models for Subject 01
-> Training CLIP model on source: schubki_cbramod
(968, 200)
   [schubki_cbramod] Train -> Top-1: 0.9370 | Top-5: 0.9959
-> Training CLIP model on source: schubki_labram
(968, 200)
   [schubki_labram] Train -> Top-1: 0.9897 | Top-5: 1.0000
-> Training CLIP model on source: schubki_luna
(968, 256)
   [schubki_luna] Train -> Top-1: 0.9690 | Top-5: 0.9990
-> Training CLIP model on source: schubki_labram_special
(968, 200)
   [schubki_labram_special] Train -> Top-1: 0.9948 | Top-5: 1.0000

🌟 Ensembled Subject 01 Train -> Top-1: 1.0000 | Top-5: 1.0000
Saved ensembled CLIP models for Subject 01!

Training Models for Subject 02
-> Training CLIP model on source: schubki_cbramod
(972, 200)
   [schubki_cbramod] Train -> Top-1: 0.8004 | Top-5: 0.9774
-> Training CLIP model on source: schubki_labram
(972, 200)
   [schubki_labram] Train -> Top-1: 0.9763 | Top-5: 1.0000
-> Training CLIP model on source: schubki_luna
(972, 256)
   [schubki_luna] Train -> Top-1: 0.9434 

In [8]:
BASE_DIR         = Path('../eeg-mind-reading-challenge')
TEST_DIR         = BASE_DIR / 'tst'
STIM_TEST_DIR    = BASE_DIR / 'stimuli_test'
FRAME_TEST_DIR   = BASE_DIR / 'frames_test'
FEAT_TEST_DIR    = BASE_DIR / 'features_test'
TEMP_FEAT        = BASE_DIR / 'temp_features'
TEST_STIM        = BASE_DIR / 'tst_stim'
ARTIFACT_DIR     = Path('../artifacts')
PREDICTIONS_DIR  = Path('../predictions')
H5_DIR           = Path('../eeg_embeddings')

os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

def features_for(ids, layer_path):
    feat_path = FEAT_TEST_DIR / f'video_{layer_path}.npz'
    z = np.load(feat_path)
    rows = []
    for sid in ids:
        key = f'{int(sid):05d}'
        assert key in z.files, f'No features found for stimulus {sid}'
        rows.append(np.array(z[key], dtype=np.float32))   
    return np.stack(rows)

global_artifact_path = ARTIFACT_DIR / 'clip_sub-global.joblib'
B = joblib.load(global_artifact_path)
sc, sc2, sc3, sc4, pca = B['sc'], B['sc2'], B['sc3'], B['sc4'], B['pca']

vids = sorted(list(STIM_TEST_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_TEST_DIR}")
all_stim_ids = np.arange(1001, 1001 + len(vids))

print("Loading all unique video features for global PCA...")
X_unique_alex = features_for(all_stim_ids, 'features.3')
X_unique_dino = features_for(all_stim_ids, 'blocks.6')
X_unique_clip = features_for(all_stim_ids, 'visual.transformer.resblocks.7')

print("Preprocessing unique video features...")
X_trn_alex_z = sc.transform(X_unique_alex)
X_trn_dino_z = sc2.transform(X_unique_dino)
X_trn_clip_z = sc3.transform(X_unique_clip)
X_trn_ensemble = np.hstack([X_trn_alex_z, X_trn_dino_z, X_trn_clip_z])
X_unique_p = pca.transform(X_trn_ensemble)
X_unique_s = sc4.transform(X_unique_p)

# Build the required lookup dictionary
video_feature_map = {int(sid): X_unique_s[i] for i, sid in enumerate(all_stim_ids)}
print("Video feature map successfully created!")

candidate_matrix = np.array([video_feature_map[int(sid)] for sid in all_stim_ids])
n_test_trials = len(all_stim_ids)
print("candidate_matrix:", candidate_matrix.shape)



Found 102 total videos in ../eeg-mind-reading-challenge/stimuli_test
Loading all unique video features for global PCA...
Preprocessing unique video features...
Video feature map successfully created!
candidate_matrix: (102, 825)


In [9]:
# --- Loop Through Each Subject for Inference ---
for sub in range(1, 7):
    bundle_path = ARTIFACT_DIR / f'clip_ensemble_sub-{sub:02d}.joblib'        
    if not bundle_path.exists():        
        print(f"\nSkipping Subject {sub:02d} (model bundle not found)")
        continue

    print(f"\n==============================")
    print(f"Testing Ensembled CLIP for Subject {sub:02d}")
    print(f"==============================")

    bundle = joblib.load(bundle_path)
    
    subject_models = bundle['models']
    sources_to_eval = bundle['sources']

    score_matrices = []

    for source_name in sources_to_eval:
        h5_test_path = H5_DIR / f'{source_name}_test/sub-{sub}.h5'
        if not os.path.exists(h5_test_path):
            print(f"-> Test H5 for source '{source_name}_test/sub-{sub}.h5' not found. Skipping this source.")
            continue

        print(f"-> Evaluating using test source: {source_name}")
        with h5py.File(h5_test_path, 'r') as f:
            X_test = np.array(f['X'], dtype=np.float32)
            
        model = subject_models[source_name]
        score_matrix = model.get_score_matrix(X_test, candidate_matrix)
        score_matrices.append(score_matrix)

    # --- SCORE-LEVEL ENSEMBLING ---
    final_score_matrix = np.mean(score_matrices, axis=0)
    
    order_idx = np.argsort(-final_score_matrix, axis=1)
    ranked = all_stim_ids[order_idx].astype(np.int32)

    # Save submission candidate list
    output_sub_path = PREDICTIONS_DIR / f'sub-{sub:02d}_ranked_candidate_list.npy'
    np.save(output_sub_path, ranked)
    print(f"Saved ensembled ranked candidate list for Subject {sub:02d} to {output_sub_path}")

    # --- Optional Evaluation if Ground Truth Exists (e.g., Subjects 1-4) ---
    gt_path = TEST_STIM / f'sub-{sub:02d}_tst_stim.npy'
    if os.path.exists(gt_path):
        gt = np.load(gt_path)
        order = np.argsort(gt)
        truth_ids = gt[order]
        
        t = np.arange(n_test_trials)
        ranks = (-final_score_matrix).argsort(1).argsort(1)[t, t] + 1
        
        top1 = (ranked[:, 0] == truth_ids).mean()
        top5 = (ranked[:, :5] == truth_ids[:, None]).any(axis=1).mean()

        print(f'Evaluated Top-1: {top1:.4f} (chance {1/n_test_trials:.4f})')
        print(f'Evaluated Top-5: {top5:.4f} (chance {5/n_test_trials:.4f})')
        print(f'Median Rank: {np.median(ranks):.1f}')
    else:
        print(f"Subject {sub:02d}: Blind candidate list generated successfully (no ground truth available).")


Testing Ensembled CLIP for Subject 01
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: schubki_labram
-> Evaluating using test source: schubki_luna
-> Evaluating using test source: schubki_labram_special
Saved ensembled ranked candidate list for Subject 01 to ../predictions/sub-01_ranked_candidate_list.npy
Evaluated Top-1: 0.0294 (chance 0.0098)
Evaluated Top-5: 0.0784 (chance 0.0490)
Median Rank: 51.5

Testing Ensembled CLIP for Subject 02
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: schubki_labram
-> Evaluating using test source: schubki_luna
-> Evaluating using test source: schubki_labram_special
Saved ensembled ranked candidate list for Subject 02 to ../predictions/sub-02_ranked_candidate_list.npy
Evaluated Top-1: 0.0000 (chance 0.0098)
Evaluated Top-5: 0.0490 (chance 0.0490)
Median Rank: 52.0

Testing Ensembled CLIP for Subject 03
-> Evaluating using test source: schubki_cbramod
-> Evaluating using test source: